# LDLR coding-variant splice site usage scoring (AlphaGenome)

Scores the predicted change in splice site usage for missense coding variants in LDLR, specified by amino-acid change (e.g. `T713G`, `Q770F`).

For each variant:
1. Look up the codon position in the LDLR CDS and map to genomic (chr19) coordinates.
2. Build the full alternate sequence with all codon substitutions applied simultaneously.
3. Score predicted splice site usage change by comparing `predict_sequence` on ref vs. alt, with a center-masked mean-delta identical to `CenterMaskScorer`.

This approach is uniform for all variants regardless of how many nucleotide positions change within the codon — there is no additivity assumption and epistatic interactions between co-occurring substitutions are captured correctly.

In [90]:
import os
import re
import numpy as np
import pandas as pd
import requests

from alphagenome.data import genome, gene_annotation
from alphagenome.data import transcript as transcript_utils
from alphagenome.models import dna_client

# Set your API key as an environment variable before launching jupyter:
#   export ALPHA_GENOME_API_KEY='your-key-here'
API_KEY = 'YOUR KEY HERE'
model = dna_client.create(API_KEY)

In [92]:
# Load GTF file containing gene and transcript locations as annotated by gencode

gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

#gtf_transcripts only keeps transcripts labeled as protein-coding
#Keep only the MANE select transcript for each gene, i.e. a consensus representative transcript per gene

# --- Filter to protein-coding MANE-select transcripts
gtf_transcripts = gene_annotation.filter_protein_coding(gtf)
gtf_transcripts = gene_annotation.filter_to_mane_select_transcript(gtf_transcripts)

# --- Initialize transcript extractor
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)

#Get interval containing LDLR
ldlr_interval = gene_annotation.get_gene_interval(gtf, gene_symbol='LDLR')

#Resize interval containing LDLR to a sequence of ~100KB centered around the gene sequence of LDLR
sequence_interval = ldlr_interval.resize(dna_client.SEQUENCE_LENGTH_100KB)
ldlr_transcripts = transcript_extractor.extract(ldlr_interval)
print(f'Extracted {len(ldlr_transcripts)} transcripts in this interval.')

#Get the MANE LDLR sequence
ldlr_tx = ldlr_transcripts[0]

print('transcript_id:', ldlr_tx.transcript_id)
print('strand:', ldlr_tx.strand, '(strand_int =', ldlr_tx.strand_int, ')')
print('num CDS exons (incl. stop codon):', len(ldlr_tx.cds_including_stop_codon))

Extracted 1 transcripts in this interval.
transcript_id: ENST00000558518.6
strand: + (strand_int = 1 )
num CDS exons (incl. stop codon): 18


In [93]:
# --- Build CDS-offset -> genomic-position (0-based) lookup table
# we're not using offset_in_cds because it doesn't do the inverse mapping, not because it was unsuitable in principle. 
# We could either (a) call it repeatedly in a brute-force inverse search, reusing it as-is but doing redundant work, or 
#(b) write a proper inverse function once, which is what the notebook does, deliberately copying its traversal logic so the two stay consistent.
# Mirrors Transcript.offset_in_cds traversal: walk cds_including_stop_codon
# in strand_int order; for - strand walk positions descending within each exon.
#In effect, this makes a dictionary {a:b}, where elements a correspond to the nucleotide offset in the LDLR codon sequence, and elements b correspond to their positions in the hg38 assembly.
#For example, the first element is {0: 11089548), where the first element 0 is the 1st nucleotide in LDLR's coding sequence and 11089548 is its position in the human genome

cds_offset_to_genomic_pos = {}
offset = 0
for cds_exon in ldlr_tx.cds_including_stop_codon[::ldlr_tx.strand_int]:
    positions = range(cds_exon.start, cds_exon.end)
    if ldlr_tx.is_negative_strand:
        positions = reversed(positions)
    for genomic_pos in positions:
        cds_offset_to_genomic_pos[offset] = genomic_pos
        offset += 1

print(f'Total CDS length (incl. stop codon): {len(cds_offset_to_genomic_pos)} bp')
print(f'({len(cds_offset_to_genomic_pos) // 3} codons incl. stop)')

Total CDS length (incl. stop codon): 2583 bp
(861 codons incl. stop)


In [94]:
# --- Reference genome sequence via UCSC REST API (hg38, 0-based half-open)
COMPLEMENT = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
_seq_cache = {}

#Fetches hg38 sequence from UCSC to determine exact DNA sequence
#Takes in chromosome, start, and end sequence. 
def get_ref_sequence(chrom, start_0based, end_0based):
    key = (chrom, start_0based, end_0based)
    if key in _seq_cache:
        return _seq_cache[key]
    url = (
        f'https://api.genome.ucsc.edu/getData/sequence?genome=hg38;'
        f'chrom={chrom};start={start_0based};end={end_0based}'
    )
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if 'dna' not in data:
        raise ValueError(f'UCSC API error for {key}: {data}')
    seq = data['dna'].upper()
    _seq_cache[key] = seq
    return seq

#Runs get_ref_sequence to determine the identify of a nucleotide (hg38 coordinates) at a given position on LDLR
def get_ref_base(genomic_pos_0based):
    return get_ref_sequence(ldlr_tx.chromosome, genomic_pos_0based, genomic_pos_0based + 1)

#Uses position of nucleotide in LDLR coding sequence to find nucleotide identity:
#Look up the hg38 coordinate given an offset in the coding sequence, call it genomic_pos
#Determine the nucleotide identity of genomic_pos using the helper function get_ref_base
#Complement the base if the gene transcript is negative strand

def transcript_base_at_cds_offset(cds_offset):
    genomic_pos = cds_offset_to_genomic_pos[cds_offset]
    ref_base = get_ref_base(genomic_pos)
    return COMPLEMENT[ref_base] if ldlr_tx.is_negative_strand else ref_base

#Convert amino acid position to position within coding DNA, returns codon bases
def codon_ref_sequence(aa_position):
    cds_offsets = [(aa_position - 1) * 3 + k for k in range(3)]
    return ''.join(transcript_base_at_cds_offset(o) for o in cds_offsets)

#Convert amino acid position to position within coding DNA, returns codon nucleotide coordinates (hg38)
def codon_to_genomic_positions(aa_position):
    """Returns 3 genomic 0-based positions in transcript 5'->3' order."""
    return [cds_offset_to_genomic_pos[(aa_position - 1) * 3 + k] for k in range(3)]

In [95]:
#Quality control to check that this dictionary is valid

# Check 1: does this say ATG?
# The logic goes that if we select the nucleotides indexed 0,1,2 in the coding sequence of LDLR, we should get the start codon ATG.
bases = [transcript_base_at_cds_offset(i) for i in range(3)]
print(''.join(bases))  # should print ATG

# Check 2: round-trip consistency against the library's own offset_in_cds
test_offset = 25
genomic_pos = cds_offset_to_genomic_pos[test_offset]
recovered = ldlr_tx.offset_in_cds(genomic_pos)
print(test_offset, recovered, test_offset == recovered)

# Check 3: reference codons vs. CSV
for aa_pos, expected in [(713, 'ACA'), (615, 'GAG'), (770, 'CAA')]:
    observed = codon_ref_sequence(aa_pos)
    status = 'Correct' if observed == expected else 'MISMATCH'
    print(f'Codon {aa_pos}: reference={observed}, CSV expects={expected}  [{status}]')

ATG
25 25 True
Codon 713: reference=ACA, CSV expects=ACA  [Correct]
Codon 615: reference=GAG, CSV expects=GAG  [Correct]
Codon 770: reference=CAA, CSV expects=CAA  [Correct]


In [96]:
# --- Parse variants CSV and build single-nt substitution lists per row
#Splice-altering-coding-variants.csv is a .csv containing information on the missense variants identified in Figure 7
#The LDLR variants are of T713, E615, and Q770

variants_df = pd.read_csv('Splice-altering-coding-variants.csv')

#Helper function to determine AA position from the first column of the .csv
#Uses regex to extract the number from each variant, e.g. "713" from T713G
def parse_aa_position(name):
    match = re.match(r'^[A-Z](\d+)[A-Z]$', name)
    if not match:
        raise ValueError(f'Could not parse variant name: {name}')
    return int(match.group(1))

#Takes one row of the .csv, extracts AA position from the "Variants induced" column
#Determines the starting and ending codon from the "Starting codon" and "Ending codon" columns
#Looks up codon at the appropriate position in the human genome and checks it against user-supplied .csv
#Looks at the nucleotides specified in starting codon and ending codon columns and check for equality at each position within the codon
#If equal, skip to the next position
#If not equal, grab genomic coordinate and specify changes to induce at individual position

def build_substitutions(row):
    """Returns list of (genomic_pos_0based, ref_plus_strand, alt_plus_strand)
    for each changed nucleotide in the codon, with ref/alt on the + strand."""
    aa_pos = parse_aa_position(row['Variants induced'])
    start_codon = row['Starting codon']
    end_codon = row['Ending codon']

    observed = codon_ref_sequence(aa_pos)
    if observed != start_codon:
        raise ValueError(
            f"{row['Variants induced']}: reference codon is {observed}, "
            f"CSV expects {start_codon}"
        )

    genomic_positions = codon_to_genomic_positions(aa_pos)
    subs = []
    for k in range(3):
        if start_codon[k] == end_codon[k]:
            continue
        gpos = genomic_positions[k]
        ref_t = start_codon[k]  # transcript-strand base
        alt_t = end_codon[k]
        if ldlr_tx.is_negative_strand:
            ref_p, alt_p = COMPLEMENT[ref_t], COMPLEMENT[alt_t]
        else:
            ref_p, alt_p = ref_t, alt_t
        actual = get_ref_base(gpos)
        if actual != ref_p:
            raise ValueError(
                f"{row['Variants induced']}: ref mismatch at {ldlr_tx.chromosome}:"
                f"{gpos+1} — computed {ref_p}, genome has {actual}"
            )
        subs.append((gpos, ref_p, alt_p))
    return aa_pos, subs

built = []
for _, row in variants_df.iterrows():
    aa_pos, subs = build_substitutions(row)
    built.append({'variant_name': row['Variants induced'], 'aa_position': aa_pos, 'subs': subs})
    sub_str = ', '.join(f'{ldlr_tx.chromosome}:{g+1} {r}->{a}' for g, r, a in subs)
    print(f"{row['Variants induced']:>6}  ->  {sub_str}")

 T713G  ->  chr19:11120519 A->G, chr19:11120520 C->G, chr19:11120521 A->C
 T713H  ->  chr19:11120519 A->C, chr19:11120520 C->A, chr19:11120521 A->C
 T713I  ->  chr19:11120520 C->T, chr19:11120521 A->C
 T713K  ->  chr19:11120520 C->A, chr19:11120521 A->G
 T713L  ->  chr19:11120519 A->C, chr19:11120520 C->T, chr19:11120521 A->G
 T713R  ->  chr19:11120520 C->G, chr19:11120521 A->G
 T713T  ->  chr19:11120521 A->G
 Q770Q  ->  chr19:11123343 A->G
 Q770E  ->  chr19:11123341 C->G, chr19:11123343 A->G
 Q770F  ->  chr19:11123341 C->T, chr19:11123342 A->T, chr19:11123343 A->C
 Q770N  ->  chr19:11123341 C->A, chr19:11123343 A->C
 Q770S  ->  chr19:11123341 C->A, chr19:11123342 A->G, chr19:11123343 A->C
 Q770V  ->  chr19:11123341 C->G, chr19:11123342 A->T, chr19:11123343 A->G


## Score joint splice site usage effect

For each variant, build the full alternate sequence with **all** codon substitutions applied simultaneously, then predict `SPLICE_SITE_USAGE` for both ref and alt sequences and compute the same center-masked mean-delta as `CenterMaskScorer` (width=501, DIFF_MEAN), centered on the midpoint of the changed positions.

This is uniform for single and multi-nucleotide codon changes — no additivity assumption, epistatic interactions captured correctly.

In [97]:
MASK_WIDTH = 501

#Input a string and return another one with desired mutations made
def apply_substitutions(sequence, interval, subs):
    """Apply (genomic_pos_0based, ref, alt) substitutions to sequence spanning interval."""
    seq_list = list(sequence)
    for gpos, ref_p, alt_p in subs:
        offset = gpos - interval.start
        if seq_list[offset] != ref_p:
            raise ValueError(
                f'Ref mismatch at offset {offset}: sequence has {seq_list[offset]}, expected {ref_p}'
            )
        seq_list[offset] = alt_p #replace the character at gpos with alt_p
    return ''.join(seq_list)

#ref_output is the reference sequence to test
#alt_output is the alternate sequence to test (with mutations)
#center_pos_0based is the genomic position that the 501 bp window should be centered around
def center_mask_diff_mean(ref_output, alt_output, center_pos_0based, width):
    """Mean of (alt - ref) splice site usage within `width` bp of center_pos_0based."""
    ref_track = ref_output.splice_site_usage #Splice site usage for reference sequence
    alt_track = alt_output.splice_site_usage #Splice site usage for alternate sequence
    track_start = ref_track.interval.start #Index of the start of the ref_track
    center_offset = center_pos_0based - track_start #Establishing the offset of the center position relative to the ref_track start
    half = width // 2 #Divide the width by 2 and drop the remainder
    lo = max(0, center_offset - half) #in practice, center_offset - 250
    hi = min(ref_track.values.shape[0], center_offset + half + 1) #in practice, center_offset + 250
    #Take the mean over the window
    return float(np.mean(alt_track.values[lo:hi] - ref_track.values[lo:hi]))
 
# Fetch reference sequence once
ref_sequence = get_ref_sequence(
    sequence_interval.chromosome, sequence_interval.start, sequence_interval.end
)
print(f'Reference sequence fetched: {len(ref_sequence)} bp')

# Reference prediction once
ref_output = model.predict_sequence(
    sequence=ref_sequence,
    interval=sequence_interval,
    requested_outputs=[dna_client.OutputType.SPLICE_SITE_USAGE],
    ontology_terms=None,
)
print('Reference prediction done.')

Reference sequence fetched: 131072 bp
Reference prediction done.


In [106]:
# This is the main scoring loop: for each variant parsed from the CSV (stored in
# `built`), build its mutant sequence, get AlphaGenome's prediction on it, and
# compare that against the single shared reference prediction (`ref_output`)
# computed once before this loop started.

results = []  # will accumulate one dict per variant; becomes the final output table

for entry in built:
    # `entry` is one variant's worth of info: {'variant_name', 'aa_position', 'subs'}
    # `subs` is the list of (genomic_pos_0based, ref_base, alt_base) tuples for this
    # variant — 1, 2, or 3 tuples depending on how many codon positions changed.
    subs = entry['subs']

    # Take the original ~131kb reference sequence and apply THIS variant's specific
    # substitution(s) to it, producing a new sequence string identical to the
    # reference everywhere except at the changed position(s). The original
    # `ref_sequence` is untouched — this returns a fresh edited copy each time.
    alt_sequence = apply_substitutions(ref_sequence, sequence_interval, subs)

    # Send the mutant sequence to AlphaGenome and get its splice-site-usage
    # prediction for it. This is a fresh API call for every variant (unlike
    # ref_output, which was computed once and is reused across all iterations).
    alt_output = model.predict_sequence(
        sequence=alt_sequence,
        interval=sequence_interval,
        requested_outputs=[dna_client.OutputType.SPLICE_SITE_USAGE],
        ontology_terms=None,
    )

    # Decide which genomic position to center the 501bp scoring window on.
    # For a single-nucleotide variant, this is just that one position.
    # For a multi-nucleotide variant (e.g. 2 or 3 changed positions within the
    # codon), this averages their positions — landing on the middle position for
    # adjacent/contiguous changes, or the midpoint of the gap for non-contiguous
    # changes (e.g. E615K, which changes codon positions 1 and 3).
    # Integer division (//) rounds down, so this is always a valid array index.
    center_pos = sum(g for g, _, _ in subs) // len(subs)

    # Compare the mutant prediction against the reference prediction within a
    # 501bp window centered on `center_pos`, exactly reproducing CenterMaskScorer's
    # windowing + DIFF_MEAN aggregation (validated empirically against the native
    # scorer earlier in this notebook). `delta` is a single number: positive means
    # the mutation is predicted to increase splice site usage nearby, negative
    # means decreased usage (e.g. consistent with disrupting a splice donor),
    # near zero means little predicted effect.
    delta = center_mask_diff_mean(ref_output, alt_output, center_pos, MASK_WIDTH)

    # Package up everything worth keeping about this variant into one dictionary
    # and add it to the running results list. Each dict will become one row of
    # the final output table.
    results.append({
        'variant_name': entry['variant_name'],
        'aa_position': entry['aa_position'],
        'n_nt_changed': len(subs),  # how many nucleotides differ in this codon change
        # Human-readable description of every substitution making up this variant,
        # e.g. "chr19:11120519 A->G; chr19:11120521 A->C" for a 2-nt change.
        # `g+1` converts the stored 0-based genomic position back to 1-based for
        # display, matching standard genomic coordinate convention.
        'substitutions': '; '.join(f'{ldlr_tx.chromosome}:{g+1} {r}->{a}' for g, r, a in subs),
        'delta_splice_usage': delta,
    })

    # Print a one-line progress update as each variant finishes, so you can watch
    # the loop progress in real time rather than waiting silently for all 16+
    # API calls to complete.
    print(f"{entry['variant_name']:>6}  delta={delta:.6f}")

# Once every variant in `built` has been processed, convert the list of result
# dictionaries into a proper pandas table: each dict becomes one row, each
# dictionary key becomes a column.
results_df = pd.DataFrame(results)

# Show the full table in the notebook output.
display(results_df)

# Save the table to disk as a CSV
results_df.to_csv('LDLR_missense_splice_scores.csv', index=False)
print('Saved LDLR_missense_splice_scores.csv')

 T713G  delta=-0.001549
 T713H  delta=-0.001091
 T713I  delta=-0.001102
 T713K  delta=-0.001504
 T713L  delta=-0.001737
 T713R  delta=-0.001719
 T713T  delta=-0.001479
 Q770Q  delta=-0.000488
 Q770E  delta=-0.000622
 Q770F  delta=-0.000331
 Q770N  delta=-0.000045
 Q770S  delta=-0.000568
 Q770V  delta=-0.000797


,variant_name,aa_position,n_nt_changed,substitutions,delta_splice_usage
0,T713G,713,3,chr19:11120519 A->G; chr19:11120520 C->G; chr1...,-0.001549
1,T713H,713,3,chr19:11120519 A->C; chr19:11120520 C->A; chr1...,-0.001091
2,T713I,713,2,chr19:11120520 C->T; chr19:11120521 A->C,-0.001102
3,T713K,713,2,chr19:11120520 C->A; chr19:11120521 A->G,-0.001504
4,T713L,713,3,chr19:11120519 A->C; chr19:11120520 C->T; chr1...,-0.001737
5,T713R,713,2,chr19:11120520 C->G; chr19:11120521 A->G,-0.001719
6,T713T,713,1,chr19:11120521 A->G,-0.001479
7,Q770Q,770,1,chr19:11123343 A->G,-0.000488
8,Q770E,770,2,chr19:11123341 C->G; chr19:11123343 A->G,-0.000622
9,Q770F,770,3,chr19:11123341 C->T; chr19:11123342 A->T; chr1...,-0.000331


Saved LDLR_missense_splice_scores.csv


In [46]:
#EXTRA FROM HERE ON DOWN

In [54]:
# from alphagenome.models import variant_scorers

# # Find a variant whose changed positions are CONTIGUOUS (so it can be expressed
# # as a single genome.Variant for comparison against the native CenterMaskScorer).
# # CenterMaskScorer/genome.Variant can handle 1+ contiguous bases; it just can't
# # handle non-contiguous changes (like E615K's positions 1+3), which is why the
# # notebook uses predict_sequence for those instead.

# def positions_are_contiguous(subs):
#     positions = sorted(g for g, _, _ in subs)
#     return positions[-1] - positions[0] + 1 == len(positions)

# contiguous_entry = next((e for e in built if positions_are_contiguous(e['subs'])), None)

# if contiguous_entry is None:
#     raise RuntimeError(
#         "No variant in `built` has contiguous changed positions — "
#         "cannot construct a comparable genome.Variant for validation."
#     )

# subs_sorted = sorted(contiguous_entry['subs'], key=lambda t: t[0])
# span_start = subs_sorted[0][0]
# ref_str = ''.join(s[1] for s in subs_sorted)
# alt_str = ''.join(s[2] for s in subs_sorted)

# print(f"Validating against variant: {contiguous_entry['variant_name']} "
#       f"({ldlr_tx.chromosome}:{span_start+1} {ref_str}->{alt_str}, "
#       f"{len(subs_sorted)} nt changed)")

# # --- Approach 1: native CenterMaskScorer via score_variant
# splice_usage_scorer = variant_scorers.CenterMaskScorer(
#     requested_output=dna_client.OutputType.SPLICE_SITE_USAGE,
#     width=MASK_WIDTH,
#     aggregation_type=variant_scorers.AggregationType.DIFF_MEAN,
# )

# native_variant = genome.Variant(
#     chromosome=ldlr_tx.chromosome,
#     position=span_start + 1,  # genome.Variant.position is 1-based
#     reference_bases=ref_str,
#     alternate_bases=alt_str,
#     name=contiguous_entry['variant_name'],
# )

# native_scores = model.score_variant(
#     interval=sequence_interval,
#     variant=native_variant,
#     variant_scorers=[splice_usage_scorer],
# )
# native_delta = float(native_scores[0].X.mean())

# # --- Approach 2: manual predict_sequence + center_mask_diff_mean (reusing ref_output)
# alt_sequence_check = apply_substitutions(
#     ref_sequence, sequence_interval, contiguous_entry['subs']
# )
# alt_output_check = model.predict_sequence(
#     sequence=alt_sequence_check,
#     interval=sequence_interval,
#     requested_outputs=[dna_client.OutputType.SPLICE_SITE_USAGE],
#     ontology_terms=None,
# )
# center_pos_check = sum(g for g, _, _ in contiguous_entry['subs']) // len(contiguous_entry['subs'])
# manual_delta = center_mask_diff_mean(ref_output, alt_output_check, center_pos_check, MASK_WIDTH)

# # --- Compare
# print(f"\nCenterMaskScorer (native):        {native_delta:.8f}")
# print(f"predict_sequence (manual):         {manual_delta:.8f}")
# print(f"Absolute difference:               {abs(native_delta - manual_delta):.2e}")

# if abs(native_delta - manual_delta) < 1e-4:
#     print("\n[OK] Manual pipeline agrees with CenterMaskScorer.")
# else:
#     print("\n[MISMATCH] Manual pipeline does NOT agree with CenterMaskScorer — "
#           "do not trust downstream results until this is resolved.")